<a href="https://colab.research.google.com/github/OlajideFemi/Carbon-Footprint/blob/main/Section6_llm_qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# section6_llm_qa.py
"""
LLM-based Quality Assurance for Government Grants Publication
Checks consistency between scheme data, award data, and bulletin narrative
"""

import pandas as pd
import json
import os
from datetime import datetime
from typing import Dict, List, Optional
import argparse

# Configure your LLM (choose one based on what you have access to)

# Option 1: OpenAI (GPT-4/GPT-3.5)
try:
    import openai
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False

# Option 2: Azure OpenAI
try:
    from openai import AzureOpenAI
    AZURE_AVAILABLE = True
except ImportError:
    AZURE_AVAILABLE = False

# Option 3: Local model via Ollama
try:
    import requests
    OLLAMA_AVAILABLE = True
except ImportError:
    OLLAMA_AVAILABLE = False


class GrantsQAChecker:
    """Quality assurance checker for government grants data"""

    def __init__(self, model_type="openai", model_name="gpt-3.5-turbo"):
        """
        Initialize the QA checker

        Args:
            model_type: "openai", "azure", "ollama", or "mock" (for testing)
            model_name: Specific model to use
        """
        self.model_type = model_type
        self.model_name = model_name
        self.client = self._initialize_client()

    def _initialize_client(self):
        """Initialize the LLM client based on model type"""
        if self.model_type == "mock":
            return None

        elif self.model_type == "openai" and OPENAI_AVAILABLE:
            openai.api_key = os.getenv("OPENAI_API_KEY")
            return openai

        elif self.model_type == "azure" and AZURE_AVAILABLE:
            return AzureOpenAI(
                api_key=os.getenv("AZURE_OPENAI_KEY"),
                api_version="2024-02-15-preview",
                azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
            )

        elif self.model_type == "ollama" and OLLAMA_AVAILABLE:
            return requests.Session()

        else:
            print(f"Warning: {self.model_type} not available. Using mock mode.")
            self.model_type = "mock"
            return None

    def summarize_schemes(self, scheme_df: pd.DataFrame, n_top: int = 10) -> str:
        """Extract key scheme information for QA checking"""
        summary = []

        # Get top schemes by value
        top_schemes = scheme_df.nlargest(n_top, 'FY value') if 'FY value' in scheme_df.columns else scheme_df.head(n_top)

        for _, scheme in top_schemes.iterrows():
            scheme_info = {
                "name": scheme.get('Scheme name', scheme.get('Grant Programme:Title', 'Unknown')),
                "funder": scheme.get('Funder: Organisation Name', scheme.get('Funding Org:Name', 'Unknown')),
                "value": f"£{scheme.get('FY value', 0)/1e9:.1f} billion" if scheme.get('FY value', 0) > 1e9 else f"£{scheme.get('FY value', 0)/1e6:.0f} million",
                "allocation_method": scheme.get('Allocation method', 'Unknown'),
                "description": str(scheme.get('Description', ''))[:200]  # Truncate long descriptions
            }
            summary.append(scheme_info)

        return json.dumps(summary, indent=2)

    def summarize_awards(self, award_df: pd.DataFrame, n_top: int = 10) -> str:
        """Extract key award information for QA checking"""
        summary = []

        # Get top awards by amount
        top_awards = award_df.nlargest(n_top, 'Amount Awarded') if 'Amount Awarded' in award_df.columns else award_df.head(n_top)

        for _, award in top_awards.iterrows():
            award_info = {
                "title": award.get('Title', award.get('Grant Award Name', 'Unknown'))[:100],
                "recipient": award.get('Recipient Org:Name', 'Unknown'),
                "amount": f"£{award.get('Amount Awarded', 0)/1e6:.1f} million" if award.get('Amount Awarded', 0) > 1e6 else f"£{award.get('Amount Awarded', 0):,.0f}",
                "scheme": award.get('Grant Programme:Title', 'Unknown'),
                "allocation_method": award.get('Allocation Method', 'Unknown')
            }
            summary.append(award_info)

        return json.dumps(summary, indent=2)

    def extract_key_figures(self, bulletin_text: str) -> Dict:
        """Extract key figures from bulletin text using LLM"""
        prompt = f"""
        Extract the following key figures from this government grants bulletin:
        1. Total grant spending (in billions)
        2. Percentage of government expenditure
        3. Number of schemes
        4. Number of awards
        5. Top department by grant value
        6. Largest scheme name and value
        7. Formula grants total and percentage
        8. General grants total and percentage

        Bulletin text:
        {bulletin_text[:8000]}  # Limit to first 8000 chars

        Return as JSON.
        """

        response = self._call_llm(prompt)

        try:
            # Try to parse JSON response
            if isinstance(response, str):
                # Extract JSON if wrapped in markdown
                if '```json' in response:
                    response = response.split('```json')[1].split('```')[0]
                figures = json.loads(response)
            else:
                figures = response
            return figures
        except:
            # Fallback to regex extraction
            return self._extract_figures_regex(bulletin_text)

    def _extract_figures_regex(self, bulletin_text: str) -> Dict:
        """Fallback: Extract figures using regex patterns"""
        import re

        figures = {}

        # Total grant spending
        total_match = re.search(r'£(\d+(?:\.\d+)?)\s*billion', bulletin_text)
        if total_match:
            figures['total_grant_spending'] = float(total_match.group(1))

        # Percentage
        pct_match = re.search(r'(\d+)%\s*of total UK government expenditure', bulletin_text)
        if pct_match:
            figures['percentage_of_expenditure'] = int(pct_match.group(1))

        return figures

    def _call_llm(self, prompt: str) -> str:
        """Call the LLM with the prompt"""
        if self.model_type == "mock":
            return self._mock_response(prompt)

        elif self.model_type == "openai":
            response = self.client.ChatCompletion.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1
            )
            return response.choices[0].message.content

        elif self.model_type == "azure":
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1
            )
            return response.choices[0].message.content

        elif self.model_type == "ollama":
            response = self.client.post(
                "http://localhost:11434/api/generate",
                json={"model": self.model_name, "prompt": prompt, "stream": False}
            )
            return response.json()["response"]

        else:
            return self._mock_response(prompt)

    def _mock_response(self, prompt: str) -> str:
        """Mock LLM response for testing"""
        return json.dumps({
            "total_grant_spending": 88.5,
            "percentage_of_expenditure": 8,
            "number_of_schemes": 450,
            "number_of_awards": 12000,
            "top_department": "Department for Education",
            "largest_scheme": "Dedicated Schools Grant",
            "formula_grants_total": 45.2,
            "general_grants_total": 43.3
        })

    def check_consistency(self, scheme_summary: str, award_summary: str, bulletin_figures: Dict) -> List[Dict]:
        """Check consistency between data sources"""
        prompt = f"""
        You are a QA checker for government grants data. Compare the following three sources and identify inconsistencies:

        SCHEME DATA (top schemes):
        {scheme_summary}

        AWARD DATA (top awards):
        {award_summary}

        BULLETIN FIGURES:
        {json.dumps(bulletin_figures, indent=2)}

        Check for:
        1. Numeric inconsistencies (e.g., totals that don't match)
        2. Naming inconsistencies (schemes mentioned in bulletin that don't appear in data)
        3. Allocation method mismatches
        4. Missing data or unexpected zeros
        5. Rounding discrepancies

        Return a list of issues with severity: "error" (must fix), "warning" (should review), or "info" (note)

        Format each issue as:
        {{
            "severity": "error|warning|info",
            "category": "numeric|naming|allocation|missing|rounding",
            "description": "clear description of the issue",
            "suggestion": "how to fix it",
            "source": "scheme|award|bulletin"
        }}
        """

        response = self._call_llm(prompt)

        try:
            if isinstance(response, str):
                # Clean response
                if '```json' in response:
                    response = response.split('```json')[1].split('```')[0]
                issues = json.loads(response)
                if isinstance(issues, dict) and 'issues' in issues:
                    issues = issues['issues']
                return issues if isinstance(issues, list) else [issues]
            return response
        except:
            return [{
                "severity": "error",
                "category": "parsing",
                "description": f"Could not parse LLM response: {response[:200]}",
                "suggestion": "Check LLM output format",
                "source": "system"
            }]

    def run_qa(self, scheme_df: pd.DataFrame, award_df: pd.DataFrame,
               bulletin_text: str, sample_n: int = 20) -> pd.DataFrame:
        """Run complete QA check"""

        print(f"Running QA check on {sample_n} samples...")

        # Summarize data
        scheme_summary = self.summarize_schemes(scheme_df, sample_n)
        award_summary = self.summarize_awards(award_df, sample_n)

        # Extract figures from bulletin
        bulletin_figures = self.extract_key_figures(bulletin_text)

        # Check consistency
        issues = self.check_consistency(scheme_summary, award_summary, bulletin_figures)

        # Convert to DataFrame
        qa_df = pd.DataFrame(issues)

        # Add timestamp
        qa_df['timestamp'] = datetime.now().isoformat()

        return qa_df


def run_qa_inline(scheme_df, award_df, bulletin_text, sample_n=20, model_type="openai"):
    """
    Inline QA function to use within notebooks

    Args:
        scheme_df: Scheme dataframe
        award_df: Award dataframe
        bulletin_text: Full bulletin text
        sample_n: Number of top items to sample
        model_type: LLM model to use ("openai", "azure", "ollama", "mock")

    Returns:
        DataFrame with QA issues
    """
    checker = GrantsQAChecker(model_type=model_type)
    return checker.run_qa(scheme_df, award_df, bulletin_text, sample_n)


def main():
    """CLI entry point"""
    parser = argparse.ArgumentParser(description='Run QA checks on government grants data')
    parser.add_argument('--scheme-file', type=str, help='Path to scheme data CSV')
    parser.add_argument('--award-file', type=str, help='Path to award data CSV')
    parser.add_argument('--bulletin-file', type=str, help='Path to bulletin text file')
    parser.add_argument('--sample-n', type=int, default=20, help='Number of items to sample')
    parser.add_argument('--model-type', type=str, default='openai', choices=['openai', 'azure', 'ollama', 'mock'])
    parser.add_argument('--output', type=str, default='qa_results.csv', help='Output CSV file')
    parser.add_argument('--fail-on-errors', action='store_true', help='Exit with error code if issues found')

    args = parser.parse_args()

    # Load data
    scheme_df = pd.read_csv(args.scheme_file) if args.scheme_file else None
    award_df = pd.read_csv(args.award_file) if args.award_file else None
    bulletin_text = open(args.bulletin_file, 'r', encoding='utf-8').read() if args.bulletin_file else ""

    # Run QA
    checker = GrantsQAChecker(model_type=args.model_type)
    qa_df = checker.run_qa(scheme_df, award_df, bulletin_text, args.sample_n)

    # Save results
    qa_df.to_csv(args.output, index=False)
    print(f"\nQA results saved to {args.output}")

    # Print summary
    print("\n" + "="*60)
    print("QA SUMMARY")
    print("="*60)
    print(f"Total issues found: {len(qa_df)}")

    if 'severity' in qa_df.columns:
        severity_counts = qa_df['severity'].value_counts()
        for severity, count in severity_counts.items():
            print(f"  {severity.upper()}: {count}")

        # Show errors
        errors = qa_df[qa_df['severity'] == 'error']
        if not errors.empty:
            print("\nERRORS (must fix before publication):")
            for _, error in errors.iterrows():
                print(f"  • {error.get('description', 'Unknown error')}")

    # Exit with error code if requested
    if args.fail_on_errors and len(qa_df[qa_df['severity'] == 'error']) > 0:
        print("\n QA failed due to errors. Please fix before publishing.")
        sys.exit(1)

    print("\n QA check completed successfully")
    return qa_df


if __name__ == "__main__":
    import sys
    main()

In [ ]:
# ===== SECTION 6: LLM-BASED QA CHECKING =====

# Import the QA module
try:
    from section6_llm_qa import run_qa_inline
    QA_AVAILABLE = True
except ImportError:
    print("Warning: section6_llm_qa module not found. Skipping QA checks.")
    QA_AVAILABLE = False

# Run QA checks
if QA_AVAILABLE:
    print("\n" + "="*60)
    print("SECTION 6: Running LLM-based QA checks")
    print("="*60)

    # Option 1: Run on sample only (faster)
    qa_df = run_qa_inline(
        scheme_df=scheme,  # Your scheme dataframe (the one you'll publish)
        award_df=award,     # Your award dataframe
        bulletin_text=full_gov_speak,  # The full bulletin text you generated
        sample_n=20,        # Check top 20 items
        model_type="openai" # or "azure", "ollama", "mock"
    )

    # Display errors that need fixing
    errors = qa_df[qa_df['severity'] == 'error']
    warnings = qa_df[qa_df['severity'] == 'warning']

    print(f"\nQA Results:")
    print(f"  Errors: {len(errors)}")
    print(f"  Warnings: {len(warnings)}")
    print(f"  Info: {len(qa_df) - len(errors) - len(warnings)}")

    if not errors.empty:
        print("\n CRITICAL ERRORS FOUND - Must fix before publication:")
        for _, error in errors.iterrows():
            print(f"  • [{error.get('category', 'unknown')}] {error.get('description', '')}")
            if 'suggestion' in error:
                print(f"    → Fix: {error['suggestion']}")

    # Save QA results
    qa_output_path = folder + '/' + blended + '/' + today + '_QA_Results.csv'
    qa_df.to_csv(qa_output_path, index=False)
    print(f"\nDetailed QA results saved to: {qa_output_path}")

    # Option 2: Run full QA (slower but more thorough)
    # Uncomment if you have time and API credits
    # qa_full_df = run_qa_inline(
    #     scheme_df=scheme,
    #     award_df=award,
    #     bulletin_text=full_gov_speak,
    #     sample_n=50,
    #     model_type="openai"
    # )

    # Halt on errors if needed
    if not errors.empty:
        print("\n  Publication halted due to QA errors.")
        print("Please fix the errors above and re-run.")
        # Uncomment to stop execution:
        # raise SystemExit(1)
else:
    print("Skipping QA checks...")

In [ ]:
# Add for OpenAI (if using)
OPENAI_API_KEY=your_openai_api_key_here

# Or for Azure OpenAI
AZURE_OPENAI_KEY=your_azure_key
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/

# Or for local Ollama
OLLAMA_HOST=http://localhost:11434

In [ ]:
# QA Configuration for Government Grants Publication

rules:
  numeric_checks:
    - description: "Scheme totals should match award totals within 5%"
      severity: error
      tolerance: 0.05

    - description: "No zero values for top 10 schemes"
      severity: error

    - description: "Values over £1bn should be labeled 'billion' consistently"
      severity: warning

  naming_checks:
    - description: "Scheme names mentioned in bulletin must exist in data"
      severity: error

    - description: "Department names must be consistent across all sources"
      severity: error

  redaction_checks:
    - description: "Do not publish fields must be properly redacted"
      severity: error

    - description: "Recipient addresses must be redacted where required"
      severity: error

  temporal_checks:
    - description: "Award dates must be within financial year"
      severity: error

    - description: "Scheme end dates must be after start dates"
      severity: error

sampling:
  top_n: 20
  random_sample: 50
  check_all_formula: true
  check_all_general: true

output:
  save_details: true
  generate_report: true
  halt_on_critical_errors: true

In [ ]:
# test_qa.py - Test the QA functionality with mock data

import pandas as pd
from section6_llm_qa import run_qa_inline

# Create mock data
mock_scheme = pd.DataFrame({
    'Scheme name': ['Test Scheme 1', 'Test Scheme 2'],
    'FY value': [1000000000, 500000000],
    'Funder: Organisation Name': ['DfE', 'DLUHC'],
    'Allocation method': ['Formula', 'General']
})

mock_award = pd.DataFrame({
    'Title': ['Test Award 1', 'Test Award 2'],
    'Amount Awarded': [50000000, 25000000],
    'Recipient Org:Name': ['Org A', 'Org B']
})

mock_bulletin = """
## Government grants statistics 2023/2024

The government spent £88.5 billion on grants. Grant spending accounts for 8% of total government expenditure.

The Department for Education gave out the greatest amount of money as grants.
"""

# Run QA
qa_results = run_qa_inline(
    scheme_df=mock_scheme,
    award_df=mock_award,
    bulletin_text=mock_bulletin,
    sample_n=10,
    model_type="mock"  # Use mock for testing
)

print("QA Results:")
print(qa_results)

In [ ]:
## Publication QA Checklist

- [ ] Run `section6_llm_qa.py` after generating bulletin
- [ ] Review all errors flagged by QA
- [ ] Fix naming inconsistencies between scheme data and bulletin
- [ ] Verify numeric totals match across sources
- [ ] Check redaction flags are correctly applied
- [ ] Validate all dates are within correct financial year
- [ ] Confirm top 10 schemes are correctly listed
- [ ] Run final QA with `--fail-on-errors` flag

In [ ]:
# After running the main script, run QA separately
python section6_llm_qa.py \
    --scheme-file "outputs/scheme_data.csv" \
    --award-file "outputs/award_data.csv" \
    --bulletin-file "outputs/final_bulletin.txt" \
    --output "outputs/qa_report.csv" \
    --fail-on-errors